# Preprocessing of application columns
- table - app_train.csv, app_valid.csv
---

## Strategy
1. Impute columns with low number of nans
2. Perform optimal bining for all variables
3. Impute the columns with complex imputing methods
4. Create capped version of numeric variables if needed

__Loan info__:
- `AMT_CREDIT` - Credit amount of the loan
- `AMT_ANNUITY` - Loan annuity
- `AMT_GOODS_PRICE` - For consumer loans it is the price of the goods for which the loan is given

## 0. Libraries and Data

In [126]:
import pandas as pd
import numpy as np

from optbinning import OptimalBinning

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from eda_module import (
    plot_quantitative_distribution, plot_binary_distribution, 
    plot_binary_vs_binary, plot_quantitative_vs_binary, plot_categorical_distribution,
    plot_categorical_vs_binary
)

from preprocess_module import (
    bin_quantitative_var, fit_optbin_var, fit_quantitative_binner, transform_quantitative_binner,
    fit_capper, transform_capper, create_imputed_quantitative_features
)

In [127]:
df_train = pd.read_csv(r"..\..\data\interim\app_train.csv")
print(f"Shape of df_train: {df_train.shape}")
df_train.head(10)

Shape of df_train: (215257, 122)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,285133,Cash loans,F,Y,Y,2,405000.0,1971072.0,68643.0,1800000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,191894,Cash loans,M,N,Y,0,337500.0,508495.5,38146.5,454500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0,0
2,369428,Cash loans,M,N,Y,1,112500.0,110146.5,13068.0,90000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,138717,Cash loans,F,N,Y,2,40500.0,66384.0,3519.0,45000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0,0
4,202381,Cash loans,M,Y,N,0,225000.0,298512.0,31801.5,270000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,185763,Cash loans,F,N,Y,0,315000.0,417024.0,21964.5,360000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0
6,148765,Cash loans,M,Y,Y,0,162000.0,168102.0,11362.5,148500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
7,448028,Cash loans,F,N,Y,0,135000.0,1129500.0,33025.5,1129500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
8,428649,Cash loans,F,N,Y,0,157500.0,1260702.0,39712.5,1129500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
9,345767,Cash loans,F,Y,N,0,180000.0,879480.0,25843.5,630000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0


In [128]:
df_valid = pd.read_csv(r"..\..\data\interim\app_valid.csv")
print(f"Shape of df_valid: {df_valid.shape}")
df_valid.head(10)

Shape of df_valid: (92254, 122)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,331034,Cash loans,F,N,Y,0,90000.0,803259.0,23616.0,670500.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0,0
1,242581,Cash loans,F,N,Y,0,270000.0,1288350.0,41692.5,1125000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,4.0,0
2,281583,Cash loans,F,N,Y,0,270000.0,253737.0,25227.0,229500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,255865,Cash loans,F,N,Y,0,144000.0,436032.0,16564.5,360000.0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,2.0,0
4,389379,Revolving loans,M,N,N,2,72000.0,225000.0,11250.0,225000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
5,455506,Cash loans,M,Y,Y,0,315000.0,521280.0,27423.0,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0,0
6,234885,Cash loans,F,Y,Y,1,54000.0,102384.0,6673.5,81000.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0
7,425217,Cash loans,M,Y,N,0,315000.0,545040.0,31288.5,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0,0
8,209875,Cash loans,F,N,Y,1,315000.0,254700.0,13567.5,225000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,106521,Cash loans,F,Y,N,0,67500.0,292500.0,13014.0,292500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0


In [129]:
df_train.dtypes.unique()

array([dtype('int64'), dtype('O'), dtype('float64')], dtype=object)

In [130]:
numeric_cols = list(df_train.select_dtypes(include=['Int64', 'float64']).columns)
cat_cols = list(df_train.select_dtypes(include=['O']).columns)

print((len(numeric_cols) + len(cat_cols)) == df_train.shape[1])

True


## 1. Impute columns with low number of nans

In [131]:
building_cols = [
    'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG',
    'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG',
    'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG',
    'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 

    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE',
    'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE',
    'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE',

    'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',

    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'
]

credit_bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]
credit_bureau_cols.remove('AMT_REQ_CREDIT_BUREAU_YEAR')

In [132]:
df_nans_train = df_train.isna().mean().to_frame(name='missing_share_train').reset_index()

df_nans_train_ge_0_05 = df_nans_train[
    (df_nans_train['missing_share_train']>=0.05) 
    & (df_nans_train['missing_share_train'] <= 0.5)
    & (~df_nans_train['index'].isin(building_cols))
    & (~df_nans_train['index'].isin(credit_bureau_cols))
    | (df_nans_train['index'] == 'EXT_SOURCE_1')
]

df_nans_valid = df_valid.isna().mean().to_frame(name='missing_share_valid').reset_index()

df_nans_valid_ge_0_05 = df_nans_valid[
    (df_nans_valid['missing_share_valid']>=0.05) 
    & (df_nans_valid['missing_share_valid'] <= 0.5)
    & (~df_nans_valid['index'].isin(building_cols))
    & (~df_nans_valid['index'].isin(credit_bureau_cols))
    | (df_nans_valid['index'] == 'EXT_SOURCE_1')
]

df_nans_ge_0_05 = df_nans_train_ge_0_05.merge(right=df_nans_valid_ge_0_05, how='left', on='index')
df_nans_ge_0_05

,index,missing_share_train,missing_share_valid
0,OCCUPATION_TYPE,0.313486,0.313385
1,EXT_SOURCE_1,0.563852,0.563715
2,EXT_SOURCE_3,0.198275,0.198203
3,AMT_REQ_CREDIT_BUREAU_YEAR,0.135099,0.134823


OCCUPATION_TYPE and AMT_REQ_CREDIT_BUREAU_YEAR will be treated as categorical so there is no need to impute them.

In [133]:
df_nans_train_l_0_05 = df_nans_train[
    (df_nans_train['missing_share_train']>0) 
    & (df_nans_train['missing_share_train'] < 0.05)
    & (~df_nans_train['index'].isin(building_cols))
    & (~df_nans_train['index'].isin(credit_bureau_cols))
]


df_nans_valid_l_0_05 = df_nans_valid[
    (df_nans_valid['missing_share_valid']>0) 
    & (df_nans_valid['missing_share_valid'] < 0.05)
    & (~df_nans_valid['index'].isin(building_cols))
    & (~df_nans_valid['index'].isin(credit_bureau_cols))
]

df_nans_l_0_05 = df_nans_train_l_0_05.merge(right=df_nans_valid_l_0_05, how='left', on='index')
df_nans_l_0_05

,index,missing_share_train,missing_share_valid
0,AMT_ANNUITY,0.000037,0.000043
1,AMT_GOODS_PRICE,0.000869,0.000986
2,NAME_TYPE_SUITE,0.004186,0.004238
3,CNT_FAM_MEMBERS,0.000005,0.000011
4,EXT_SOURCE_2,0.002156,0.002125
5,OBS_30_CNT_SOCIAL_CIRCLE,0.003317,0.003328
6,DEF_30_CNT_SOCIAL_CIRCLE,0.003317,0.003328
7,OBS_60_CNT_SOCIAL_CIRCLE,0.003317,0.003328
8,DEF_60_CNT_SOCIAL_CIRCLE,0.003317,0.003328
9,DAYS_LAST_PHONE_CHANGE,0.000005,NaN


In [134]:
cols_to_simple_impute = list(df_nans_l_0_05['index'].values)
cols_to_simple_impute

['AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'CNT_FAM_MEMBERS',
 'EXT_SOURCE_2',
 'OBS_30_CNT_SOCIAL_CIRCLE',
 'DEF_30_CNT_SOCIAL_CIRCLE',
 'OBS_60_CNT_SOCIAL_CIRCLE',
 'DEF_60_CNT_SOCIAL_CIRCLE',
 'DAYS_LAST_PHONE_CHANGE']

In [135]:
print(f"df_train: {df_train.shape}")
print(f"df_valid: {df_valid.shape}")

df_train: (215257, 122)
df_valid: (92254, 122)


In [136]:
df_train1 = df_train.copy()
df_valid1 = df_valid.copy()

for col in cols_to_simple_impute:
    if col in numeric_cols:
        imp_value = df_train[col].median()
        
        df_train1[f"{col}_missing"] = np.where(df_train1[col].isna(), 1, 0)
        df_valid1[f"{col}_missing"] = np.where(df_valid1[col].isna(), 1, 0)
        
        df_train1[f"{col}_imp_global"] = imp_value
        df_valid1[f"{col}_imp_global"] = imp_value

    if col in cat_cols:
        imp_value = df_train1[col].mode()[0]

        df_train1[f"{col}_missing"] = np.where(df_train1[col].isna(), 1, 0)
        df_valid1[f"{col}_missing"] = np.where(df_valid1[col].isna(), 1, 0)
        
        df_train1[f"{col}_imp_global"] = imp_value
        df_valid1[f"{col}_imp_global"] = imp_value

In [137]:
numeric_cols1 = list(df_train1.select_dtypes(include=['Int64', 'float64']).columns)
cat_cols1 = list(df_train1.select_dtypes(include=['O']).columns)

print((len(numeric_cols1) + len(cat_cols1)) == df_train1.shape[1])

True


In [138]:
print(f"df_train1: {df_train1.shape}")
print(f"df_valid1: {df_valid1.shape}")

df_train1: (215257, 142)
df_valid1: (92254, 142)


## 2. Perform optimal binning for all variables

In [139]:
df_train2 = df_train1.copy()
df_valid2 = df_valid1.copy()

for n, col in enumerate(df_train.columns):

    # if n >= 1:
    #     summary = summary2.copy()

    df_train2, df_valid2, optb2, summary2 = fit_optbin_var(
        df_train=df_train2,
        df_valid=df_valid2,
        var=col,
        dtype='numerical' if col in numeric_cols1 else 'categorical',
        target='TARGET',
        metric='woe',
        monotonic_trend='auto' if col in numeric_cols1 else None,
        min_bin_size=0.05,
        special_codes=[365243] if col=='DAYS_EMPLOYED' else None
    )

    # if n >= 1:
    #     summary = summary2.concat()



In [140]:
print(f"df_train2: {df_train2.shape}")
print(f"df_valid2: {df_valid2.shape}")

df_train2: (215257, 264)
df_valid2: (92254, 264)


In [141]:
df_train2.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19_woe,FLAG_DOCUMENT_20_woe,FLAG_DOCUMENT_21_woe,AMT_REQ_CREDIT_BUREAU_HOUR_woe,AMT_REQ_CREDIT_BUREAU_DAY_woe,AMT_REQ_CREDIT_BUREAU_WEEK_woe,AMT_REQ_CREDIT_BUREAU_MON_woe,AMT_REQ_CREDIT_BUREAU_QRT_woe,AMT_REQ_CREDIT_BUREAU_YEAR_woe,TARGET_woe
0,285133,Cash loans,F,Y,Y,2,405000.0,1971072.0,68643.0,1800000.0,...,2.220446e-16,2.220446e-16,2.220446e-16,0.051806,0.051806,0.051806,0.043319,0.032891,0.134771,2.220446e-16
1,191894,Cash loans,M,N,Y,0,337500.0,508495.5,38146.5,454500.0,...,2.220446e-16,2.220446e-16,2.220446e-16,0.051806,0.051806,0.051806,0.043319,0.032891,-0.083169,2.220446e-16
2,369428,Cash loans,M,N,Y,1,112500.0,110146.5,13068.0,90000.0,...,2.220446e-16,2.220446e-16,2.220446e-16,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.220446e-16
3,138717,Cash loans,F,N,Y,2,40500.0,66384.0,3519.0,45000.0,...,2.220446e-16,2.220446e-16,2.220446e-16,0.051806,0.051806,0.051806,0.096105,0.032891,0.007111,2.220446e-16
4,202381,Cash loans,M,Y,N,0,225000.0,298512.0,31801.5,270000.0,...,2.220446e-16,2.220446e-16,2.220446e-16,0.051806,0.051806,0.051806,0.043319,0.032891,0.134771,2.220446e-16


In [142]:
for n, col in enumerate(df_train2.columns):
    print(f"{n}. {col}")

0. SK_ID_CURR
1. NAME_CONTRACT_TYPE
2. CODE_GENDER
3. FLAG_OWN_CAR
4. FLAG_OWN_REALTY
5. CNT_CHILDREN
6. AMT_INCOME_TOTAL
7. AMT_CREDIT
8. AMT_ANNUITY
9. AMT_GOODS_PRICE
10. NAME_TYPE_SUITE
11. NAME_INCOME_TYPE
12. NAME_EDUCATION_TYPE
13. NAME_FAMILY_STATUS
14. NAME_HOUSING_TYPE
15. REGION_POPULATION_RELATIVE
16. DAYS_BIRTH
17. DAYS_EMPLOYED
18. DAYS_REGISTRATION
19. DAYS_ID_PUBLISH
20. OWN_CAR_AGE
21. FLAG_MOBIL
22. FLAG_EMP_PHONE
23. FLAG_WORK_PHONE
24. FLAG_CONT_MOBILE
25. FLAG_PHONE
26. FLAG_EMAIL
27. OCCUPATION_TYPE
28. CNT_FAM_MEMBERS
29. REGION_RATING_CLIENT
30. REGION_RATING_CLIENT_W_CITY
31. WEEKDAY_APPR_PROCESS_START
32. HOUR_APPR_PROCESS_START
33. REG_REGION_NOT_LIVE_REGION
34. REG_REGION_NOT_WORK_REGION
35. LIVE_REGION_NOT_WORK_REGION
36. REG_CITY_NOT_LIVE_CITY
37. REG_CITY_NOT_WORK_CITY
38. LIVE_CITY_NOT_WORK_CITY
39. ORGANIZATION_TYPE
40. EXT_SOURCE_1
41. EXT_SOURCE_2
42. EXT_SOURCE_3
43. APARTMENTS_AVG
44. BASEMENTAREA_AVG
45. YEARS_BEGINEXPLUATATION_AVG
46. YEARS_BUILD_

In [143]:
df_train2_bins = df_train1.copy()
df_valid2_bins = df_valid1.copy()

for n, col in enumerate(df_train.columns):

    # if n >= 1:
    #     summary = summary2.copy()

    df_train2_bins, df_valid2_bins, optb2, summary2 = fit_optbin_var(
        df_train=df_train2,
        df_valid=df_valid2,
        var=col,
        dtype='numerical' if col in numeric_cols1 else 'categorical',
        target='TARGET',
        metric='bins',
        monotonic_trend='auto' if col in numeric_cols1 else None,
        min_bin_size=0.05,
        special_codes=[365243] if col=='DAYS_EMPLOYED' else None
    )

## 3. Impute the columns with complex imputing methods

### 3.1 EXT_SOURCE_1

In [144]:
df_train3_1 = df_train2.copy()
df_valid3_1 = df_valid2.copy()

In [145]:
print(f"df_train3_1: {df_train3_1.shape}")
print(f"df_valid3_1: {df_valid3_1.shape}")

df_train3_1: (215257, 264)
df_valid3_1: (92254, 264)


In [146]:
df_train3_1['EXT_SOURCE_1_missing'] = np.where(df_train3_1['EXT_SOURCE_2'].isna(), 1, 0)
df_valid3_1['EXT_SOURCE_1_missing'] = np.where(df_valid3_1['EXT_SOURCE_2'].isna(), 1, 0)

In [147]:
df_train3_1['NAME_INCOME_TYPE_woe_obj'] = df_train3_1['NAME_INCOME_TYPE_woe'].astype('object')
df_valid3_1['NAME_INCOME_TYPE_woe_obj'] = df_valid3_1['NAME_INCOME_TYPE_woe'].astype('object')

In [148]:
ext_1_imputation_spec = {
    "EXT_SOURCE_1_imp_income_type_gender": {
        "metod": "median",
        "group_levels": [['NAME_INCOME_TYPE_woe_obj', 'CODE_GENDER'], ['NAME_INCOME_TYPE_woe_obj']]
    },
}

df_train3_1, df_valid3_1, ext_1_summary = create_imputed_quantitative_features(
    df=df_train3_1,
    df_valid=df_valid3_1,
    value_col='EXT_SOURCE_1',
    specs=ext_1_imputation_spec,
    add_invalid_flag=False,
    add_clean_col=False,
    return_summary=True
)

df_train3_1 = df_train3_1.drop(columns=['NAME_INCOME_TYPE_woe_obj'])
df_valid3_1 = df_valid3_1.drop(columns=['NAME_INCOME_TYPE_woe_obj'])

ext_1_summary


,dataset,output_col,method,n_invalid,missing_before_imputation,filled_by_hierarchy,filled_by_global,missing_after_imputation,group_levels,step,group_level,filled_count
0,train,EXT_SOURCE_1_imp_income_type_gender,median,0.0,121373.0,121373.0,0.0,0.0,"[['NAME_INCOME_TYPE_woe_obj', 'CODE_GENDER'], ...",NaN,NaN,NaN
1,train,EXT_SOURCE_1_imp_income_type_gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"(NAME_INCOME_TYPE_woe_obj, CODE_GENDER)",121372.0
2,train,EXT_SOURCE_1_imp_income_type_gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,"(NAME_INCOME_TYPE_woe_obj,)",1.0
3,valid,EXT_SOURCE_1_imp_income_type_gender,median,0.0,52005.0,52005.0,0.0,0.0,"[['NAME_INCOME_TYPE_woe_obj', 'CODE_GENDER'], ...",NaN,NaN,NaN
4,valid,EXT_SOURCE_1_imp_income_type_gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"(NAME_INCOME_TYPE_woe_obj, CODE_GENDER)",52004.0
5,valid,EXT_SOURCE_1_imp_income_type_gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,"(NAME_INCOME_TYPE_woe_obj,)",1.0


In [149]:
print(f"df_train3_1: {df_train3_1.shape}")
print(f"df_valid3_1: {df_valid3_1.shape}")

df_train3_1: (215257, 266)
df_valid3_1: (92254, 266)


### 3.2 EXT_SOURCE_2

In [158]:
df_train3_2 = df_train3_1.copy()
df_valid3_2 = df_valid3_1.copy()

In [159]:
df_train3_2['EXT_SOURCE_3_missing'] = np.where(df_train3_2['EXT_SOURCE_3'].isna(), 1, 0)
df_valid3_2['EXT_SOURCE_3_missing'] = np.where(df_valid3_2['EXT_SOURCE_3'].isna(), 1, 0)

In [160]:
df_train3_2['NAME_EDUCATION_TYPE_woe_obj'] = df_train3_2['NAME_EDUCATION_TYPE_woe'].astype('object')
df_valid3_2['NAME_EDUCATION_TYPE_woe_obj'] = df_valid3_2['NAME_EDUCATION_TYPE_woe'].astype('object')

In [161]:
ext_3_imputation_spec = {
    "EXT_SOURCE_3_imp_education": {
        "metod": "median",
        "group_levels": [['NAME_EDUCATION_TYPE_woe_obj']]
    },
}

df_train3_2, df_valid3_2, ext_3_summary = create_imputed_quantitative_features(
    df=df_train3_2,
    df_valid=df_valid3_2,
    value_col='EXT_SOURCE_3',
    specs=ext_3_imputation_spec,
    add_invalid_flag=False,
    add_clean_col=False,
    return_summary=True
)

df_train3_2 = df_train3_2.drop(columns=['NAME_EDUCATION_TYPE_woe_obj'])
df_valid3_2 = df_valid3_2.drop(columns=['NAME_EDUCATION_TYPE_woe_obj'])

ext_3_summary

,dataset,output_col,method,n_invalid,missing_before_imputation,filled_by_hierarchy,filled_by_global,missing_after_imputation,group_levels,step,group_level,filled_count
0,train,EXT_SOURCE_3_imp_education,median,0.0,42680.0,42680.0,0.0,0.0,[['NAME_EDUCATION_TYPE_woe_obj']],NaN,NaN,NaN
1,train,EXT_SOURCE_3_imp_education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"(NAME_EDUCATION_TYPE_woe_obj,)",42680.0
2,valid,EXT_SOURCE_3_imp_education,median,0.0,18285.0,18285.0,0.0,0.0,[['NAME_EDUCATION_TYPE_woe_obj']],NaN,NaN,NaN
3,valid,EXT_SOURCE_3_imp_education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"(NAME_EDUCATION_TYPE_woe_obj,)",18285.0


In [162]:
print(f"df_train3_2: {df_train3_2.shape}")
print(f"df_valid3_2: {df_valid3_2.shape}")

df_train3_2: (215257, 268)
df_valid3_2: (92254, 268)


## 4. Saving processed data sets

In [163]:
PROJECT_PATH = Path.cwd().parent.parent
DATA_SAVE_PATH = PROJECT_PATH / 'data' / 'interim'

if str(DATA_SAVE_PATH) not in sys.path:
    sys.path.insert(0, str(DATA_SAVE_PATH))

DATA_SAVE_PATH.mkdir(parents=True, exist_ok=True)

df_train3_2.to_csv(DATA_SAVE_PATH / 'df_train_processed.csv', index=False)
print(f"df_train3_2 saved to: DATA_SAVE_PATH\\df_train_processed.csv")

df_valid3_2.to_csv(DATA_SAVE_PATH / 'df_valid_processed.csv', index=False)
print(f"df_valid3_2 saved to: DATA_SAVE_PATH\\df_valid_processed.csv")

df_train3_2 saved to: DATA_SAVE_PATH\df_train_processed.csv
df_valid3_2 saved to: DATA_SAVE_PATH\df_valid_processed.csv
